# <span style = "color:rebeccapurple"> Classification

<span style="text-transform: uppercase;
        font-size: 14px;
        letter-spacing: 1px;
        font-family: 'Segoe UI', sans-serif;">
    Author
</span><br>
efrén cruz cortés
<hr style="border: none; height: 1px; background: linear-gradient(to right, transparent 0%, #ccc 10%, transparent 100%); margin-top: 10px;">

## <span style = "color:darkorange"> Conceptual Intermezzo - What is classification?

See slides

## <span style = "color:darkorchid"> Imports

In [ ]:
# :: IMPORTS ::

# Scikit-learn specifics:
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn import svm

# for digits exercise:
from sklearn import datasets

# Helper modules
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# :: Data ::

try:
    import google.colab
    !wget https://raw.githubusercontent.com/nuitrcs/machine-learning-2026/refs/heads/main/data/penguins.csv
    !wget https://raw.githubusercontent.com/nuitrcs/machine-learning-2026/refs/heads/main/data/fish.csv
    !wget https://raw.githubusercontent.com/nuitrcs/machine-learning-2026/refs/heads/main/data/wine.csv
    penguins_directory = "penguins.csv"
    fish_directory = "fish.csv"
    wine_directory = "wine.csv"
    print("Successfully loaded files to Colab. Check folder on left column.")
except ModuleNotFoundError:
    penguins_directory = "data/penguins.csv"
    fish_directory = "data/fish.csv"
    wine_directory = "data/wine.csv"
    print("Data should be in your local directory. Under the 'data' folder.")

Now that we are experienced with preprocessing and pipelines. Let's move on to a different machine learning task: classification.

## <span style = "color:darkorchid"> Alice: sommelière extraordinaire!

After coming back from her cold polar adventures, Alice is ready for a change of scenery, so she takes a trip to Italy. Being quite the sommelière herself, she visits her favorite ristorante in La Toscana, *Il Cappellaio Matto*, in search of good wine. As it turns out, there is a heated debate going on, everyone is wondering if sommelieres can actually distinguish different wines, or if they are just faking it.

Alice decides to settle the dispute with machine learning. Her colleagues trust her, so they give her a dataset containing the chemical composition of different wine samples. All these samples were grown in the Toscana region, but they were made of three different cultivars (a cultivar is a specific plant variety, in this case varieties of grape vines).

![alice-sommerlier](images/alice_sommelier.png){width=30%}

### <span style = "color:teal"> Load data

In [ ]:
wine = pd.read_csv(wine_directory)

In [ ]:
wine_X = wine.drop(columns=['target'])
wine_y = wine["target"]

In [ ]:
wine_X.shape

In [ ]:
wine_X.head()

In [ ]:
wine_y.head()

<b>NOTE</b> scikit learn classifiers usually expect the target to be a 1-d array or a series, so we'll keep it as a series.

And just to make sure we don't mess things up, let's load it again in a unified cell:

In [ ]:
# Load wine data
wine = pd.read_csv(wine_directory)
wine_X = wine.drop(columns=['target'])
wine_y = wine["target"]

### <span style = "color:teal"> Using a support vector classifier

For now, we will skip the preprocessing stage and head straight to the classification task. `scikit-learn` provides different classifiers for us, let's start with the support vector machine (SVM). You can see the documentation [here](https://scikit-learn.org/stable/modules/svm.html). We can find support vector classifiers in the `svm` module, which we have already imported.

In [ ]:
wX_train, wX_test, wy_train, wy_test = train_test_split(wine_X, wine_y, test_size = .15)

In [ ]:
# Create SVM classifier:
wine_clf = svm.SVC()

In [ ]:
# let's see our classifier
wine_clf

In [ ]:
# Fit the SVM classifier:
wine_clf.fit(wX_train, wy_train)

In [ ]:
wine_clf.classes_

In [ ]:
# Predict the cultivars
wine_predictions = wine_clf.predict(wX_test)
wine_predictions

Let's compare the predictions to the true values:

In [ ]:
comparison_df = pd.DataFrame(data = {"Predicted": wine_predictions,
                                    "True Cultivars": wy_test})

In [ ]:
comparison_df.head(20)

Note: we didn't have a lot of data points, and since we have three classes (wine varieties), we are a bit tight on our data. That's why I chose a small test size. If you go back and change the test size parameter to something like $.3$, you will notice a big drop in performance.

Let's check now our accuracy (the proportion of times we got it right):

In [ ]:
# Evaluate:
wine_clf.score(wX_test, wy_test)

### <span style = "color:teal"> Adding Preprocessing

As stated before, the type of preprocessing we must do depends on your data and your model. All our features are numeric, which makes things easier. Let's go with the standard scaler for now.

In [ ]:
# Create and fit preprocessor
wine_preprocessor = preprocessing.StandardScaler().fit(wX_train, wy_train)

In [ ]:
# Transform data to standard scale
wX_train_trans = wine_preprocessor.transform(wX_train)

In [ ]:
# Fit classifier to transformed data
wine_clf = svm.SVC().fit(wX_train_trans, wy_train)

In [ ]:
# Evaluate on testing data:
    # Transform test data
wX_test_trans = wine_preprocessor.transform(wX_test)

    # Check accuracy
wine_clf.score(wX_test_trans, wy_test)

Woah!! Our accuracy skyrocketed, looks like preprocessing is very important uh?

### <span style = "color:teal"> Make a pipeline

In [ ]:
# Step 1: Load the data
wine = pd.read_csv(wine_directory)
wine_X = wine.drop(columns=['target'])
wine_y = wine["target"]

# Step 2: Split the data
seed = 42
wX_train, wX_test, wy_train, wy_test = train_test_split(wine_X, wine_y, test_size = .3, random_state=seed)

# Step 3: Create the pipeline
wine_pipeline = Pipeline(
    [
        ("preprocessor", preprocessing.StandardScaler()),
        ("classifier", svm.SVC())
    ]
)

# Step 4: Fit the pipeline
wine_pipeline.fit(wX_train, wy_train)

# Step 4.5: You can predict the class based on newly observed features:
print(wine_pipeline.predict(wX_train[0:3]))     # <-- Predict class for first 3 test datapoints

# Step 5: You can evaluate the accuracy of the classifier on the whole test dataset
wine_pipeline.score(wX_test, wy_test)

In [ ]:
wine_pipeline

### <span style = "color:teal"> Visualization and Intuition

**Bo's Field Data Collection**

Bo is quite the plant enthusiast, and decides to collect data from different iris species. They wonder if they can guess the particular species based only on measurements of widths and lengths of the sepals and petals.

![bo-iris](images/bo_iris_cropped.png){width=35%}

![iris-species](images/iris.png){width=50%}

Below I'm plotting two features of the iris dataset (we'll get back to it in a later section), together with the classification regions. The colored regions is what the SVM decides to classify as one class or another. Each point is colored by its true label.

(The code for generating this image is found in `support_materials.ipynb`)
<br>
<br>

![classification-visualization](images/classification_viz.png){width=50%}

## <span style = "color:red"> Long Exercise - Bo's daunting dilemma

Having followed Alice to Italy, Bo travels south to visit the famous city of Pompei. In there, they find the walls are full of ancient romans' graffiti! Bo would like to have a dataset of all the text in the walls, but copying it one by one would be incredibly daunting. They instead decide to take pictures and figure it out later.

Back in their lab, Bo needs to create a classifier that takes as inputs images of hand-written text, and maps them to their proper symbol. To begin, Bo will focus on numerical digits.

Load the "digits" dataset from `scikit-learn` and build a classifier to achieve this task.

![pompeii-graffiti](images/pompeii_02.png){width=40%}

Good luck!

**Notes**

- Each data point is an 8x8 pixels image
- An image is basically a matrix of numbers, where numbers at each position represent color values (grey scale)
- You can "flatten" this matrix into a long vector, which will be your feature vector
- The labels are the actual characters for the different numbers (0, 1, 2, etc.)
- Your dataset will contain 'images', which is a 3D array representing all images (N, 8, 8)
- Your dataset will contain 'data', which is the flattened array (N, 64)

In [ ]:
# Load digits dataset
digits = datasets.load_digits()

# Print the description of the dataset
print(digits['DESCR'])

In [ ]:
# Let me show you an image, in numeric matrix form
digits.images[0]

In [ ]:
# This is the flattened version, use it for classification
digits.data[0]

In [ ]:
# This is how the image actually looks, can you guess the digit?
plt.imshow(digits.images[42], cmap='Greys')
plt.show()
print(f"The digit is: {digits.target[42]}")

In [ ]:
# Now build your own classification pipeline!
# Hint: your predictor X is inside digits.data, and your target y in digits.target


# <span style = "color:rebeccapurple"> Hyperparameters

Many machine learning algorithms depend on a few hyperparameters, which can be very influential for performance. Let's talk a bit about the hyperparameters for the support vector classifier.

**Regularization.** Regularization is the process of penalizing complicated decision boundaries. For the `SVC` class, this is done through the parameter `C`.

**Local Influence.** The extent to which individual points influence the classifier can be controlled with the parameter `gamma`. If gamma is too large, training data points will not interact much when making up the classifier, effectively creating pockets of one class. If on the other hand, gamma is too small, the influence of points spread throughout space, effectively creating a large region for one class.

Let's see how they affect performance to build an intuition. The default value of `C` is $1$. The default value of `gamma` is a heuristic based on the data's variance.

In [ ]:
# let's make a function that takes in a value of C, and a dataset, and outputs the classification accuracy
def full_classification(data_X, data_y, C_param, gamma_param = None):
    if gamma_param:
        params = {'C': C_param, 'gamma':gamma_param}
    else:
        params = {'C': C_param}
    seed = 42
    X_train, X_test, y_train, y_test = train_test_split(data_X, data_y, test_size = .25, random_state=seed)
    classif_pipeline = Pipeline([
        ("preprocessor", preprocessing.StandardScaler()),
        ("classifier", svm.SVC(**params))
    ])
    classif_pipeline.fit(X_train, y_train)
    accuracy = classif_pipeline.score(X_test, y_test)
    return accuracy

In [ ]:
# Load the data
wine = pd.read_csv(wine_directory)
wine_X = wine.drop(columns=['target'])
wine_y = wine['target']

# Get accuracy for different levels of C
C_vals = [0.01, .001, .5, 1, 10, 100, 1000]
accs = []
for c in C_vals:
    accs.append(full_classification(wine_X, wine_y, c))
print(accs)

Let's test `gamma` now:

In [ ]:
# Load the data
wine = pd.read_csv(wine_directory)

# Get accuracy for different levels of gamma
gamma_vals = [.001, .01, .1, .5, 1, 10, 100]
accs = []
for g in gamma_vals:
    accs.append(full_classification(wine_X, wine_y, C_param=1, gamma_param=g))
print(accs)

We'll get back to hyperparameters in the next two sections, when we look at evaluation and cross-validation.

## <span style="color:darkorchid">Other Classifiers</span>

Finally, while we do not have time to go over other classification algorithms, `sklearn` offers a variety, including nearest neighbors and random forests. For more details refer to the [documentation](https://scikit-learn.org/stable/supervised_learning.html).